# Chapter 9 · Evaluating Diagnostic Tests and Screening: companion notebook
*Data Science and Artificial Intelligence in Health, Volume 1*

> **This notebook accompanies the book chapter; it does not reproduce it.** Definitions, derivations, and explanations are in the book. Here you run the calculations on the case and check the numbers. All data are **synthetic**, and the values are for teaching purposes only.

**Part A** follows the chapter's case, the evaluation of a rapid test and of a wound biomarker for surgical-site infection, through the quality directorate's questions; only the sections with a calculation appear (9.3 to 9.7). The book draws the figures; the notebook prints the same numbers. **Part B** gives the written solution of every chapter exercise (1 to 12; statements are in the book) and, for the exercises whose solution needs a calculation (3, 4, 7, 8, 9, 11, and 12), its code and result. The code is Python 3 with pandas, statsmodels, scikit-learn, and scipy. Run the cells in order.

## Part A. The case

**Data.** `data/accuracy_cohort.csv`: 1,200 consecutive 30-day follow-up visits, with the wound biomarker (`marker_ng_ml`), the rapid test (`rapid_test`), the true infection (`ssi`, the reference standard), and other fields. `data/two_gate.csv`: 100 deep infections and 100 healthy volunteers. `data/screening_cohort.csv`: 10,000 women followed for 20 years, with the times of preclinical onset, clinical diagnosis, and death. Each cell carries the number of the book section it belongs to.

### 9.3 The two-by-two table of a test: sensitivity and specificity

*How well does the rapid test detect infection and rule it out?*

In [1]:
# 1. Read the accuracy cohort: 1,200 follow-up visits, with the rapid test (index test) and the true infection (reference standard)
import pandas as pd
from statsmodels.stats.proportion import proportion_confint

d = pd.read_csv("data/accuracy_cohort.csv")
inf = d["ssi"] == "yes"
pos = d["rapid_test"] == "positive"

# 2. The four cells of the two-by-two table: a Boolean column summed is a count
tp = (pos & inf).sum()
fn = (~pos & inf).sum()
fp = (pos & ~inf).sum()
tn = (~pos & ~inf).sum()
print("True positives:", tp, "; false positives:", fp, "; false negatives:", fn, "; true negatives:", tn)

# 3. Sensitivity, specificity, and the predictive values
se = tp / (tp + fn)
sp = tn / (tn + fp)
ppv = tp / (tp + fp)
npv = tn / (tn + fn)
print("Sensitivity:", round(se, 3), "; specificity:", round(sp, 3), "; positive predictive value:", round(ppv, 3), "; negative predictive value:", round(npv, 3))

# 4. Wilson 95% intervals of the four proportions
for name, k, n in [("Sensitivity", tp, tp + fn), ("Specificity", tn, tn + fp), ("Positive predictive value", tp, tp + fp), ("Negative predictive value", tn, tn + fn)]:
    lo, hi = proportion_confint(k, n, alpha=0.05, method="wilson")
    print(name + ":", round(k / n, 3), "(95% CI", round(lo, 3), "to", round(hi, 3), ")")

True positives: 67 ; false positives: 193 ; false negatives: 27 ; true negatives: 913
Sensitivity: 0.713 ; specificity: 0.825 ; positive predictive value: 0.258 ; negative predictive value: 0.971
Sensitivity: 0.713 (95% CI 0.614 to 0.794 )
Specificity: 0.825 (95% CI 0.802 to 0.847 )
Positive predictive value: 0.258 (95% CI 0.208 to 0.314 )
Negative predictive value: 0.971 (95% CI 0.959 to 0.98 )


### 9.4 Predictive values and prevalence

*What does a positive or a negative result mean at different prevalences?*

In [2]:
# 1. Predictive values of the rapid test at different prevalences, from its sensitivity and specificity
for prev in [0.01, 0.03, 0.078, 0.20, 0.40]:
    ppv_prev = prev * se / (prev * se + (1 - prev) * (1 - sp))
    npv_prev = (1 - prev) * sp / ((1 - prev) * sp + prev * (1 - se))
    print("Prevalence", prev, ": positive predictive value", round(ppv_prev, 3), "; negative predictive value", round(npv_prev, 3))

Prevalence 0.01 : positive predictive value 0.04 ; negative predictive value 0.996
Prevalence 0.03 : positive predictive value 0.112 ; negative predictive value 0.989
Prevalence 0.078 : positive predictive value 0.257 ; negative predictive value 0.971
Prevalence 0.2 : positive predictive value 0.505 ; negative predictive value 0.92
Prevalence 0.4 : positive predictive value 0.731 ; negative predictive value 0.812


### 9.5 Likelihood ratios

*How much does each result change the odds of infection?*

In [3]:
# 1. The likelihood ratios of the binary test
lr_pos = se / (1 - sp)
lr_neg = (1 - se) / sp
print("Positive likelihood ratio:", round(lr_pos, 2), "; negative likelihood ratio:", round(lr_neg, 2))

# 2. The likelihood ratio of each band of the biomarker (ng/mL), from the counts of infected and not infected patients in the band
bands = [(0, 20), (20, 40), (40, 80), (80, float("inf"))]
n_inf = inf.sum()
n_not = (~inf).sum()
for lo, hi in bands:
    in_band = (d["marker_ng_ml"] >= lo) & (d["marker_ng_ml"] < hi)
    a = (in_band & inf).sum()
    b = (in_band & ~inf).sum()
    print("Band", lo, "to", hi, ": infected", a, "; not infected", b, "; proportions", round(a / n_inf, 3), round(b / n_not, 3), "; likelihood ratio", round((a / n_inf) / (b / n_not), 2))

Positive likelihood ratio: 4.08 ; negative likelihood ratio: 0.35
Band 0 to 20 : infected 10 ; not infected 663 ; proportions 0.106 0.599 ; likelihood ratio 0.18
Band 20 to 40 : infected 27 ; not infected 362 ; proportions 0.287 0.327 ; likelihood ratio 0.88
Band 40 to 80 : infected 37 ; not infected 75 ; proportions 0.394 0.068 ; likelihood ratio 5.8
Band 80 to inf : infected 20 ; not infected 6 ; proportions 0.213 0.005 ; likelihood ratio 39.22


### 9.6 From pre-test to post-test probability: natural frequencies and odds

*What is the probability of infection after the test, starting from the probability before it?*

In [4]:
# 1. Natural frequencies: 1,000 patients at a pre-test probability of 7.8%
n = 1000
infected = round(n * 0.078)
print("Infected:", infected, "; true positives:", round(infected * se), "; false positives:", round((n - infected) * (1 - sp)))

# 2. Post-test probability from the pre-test probability and a likelihood ratio, in the odds form
cases = [("7.8%, positive strip", 0.078, lr_pos), ("7.8%, negative strip", 0.078, lr_neg),
         ("7.8%, top band", 0.078, 39.22), ("7.8%, bottom band", 0.078, 0.18),
         ("30%, positive strip", 0.30, lr_pos), ("30%, negative strip", 0.30, lr_neg)]
for label, prev, lr in cases:
    pre_odds = prev / (1 - prev)
    post_odds = pre_odds * lr
    print("Pre-test", label + ": post-test probability", round(post_odds / (1 + post_odds), 3))

Infected: 78 ; true positives: 56 ; false positives: 161
Pre-test 7.8%, positive strip: post-test probability 0.257
Pre-test 7.8%, negative strip: post-test probability 0.029
Pre-test 7.8%, top band: post-test probability 0.768
Pre-test 7.8%, bottom band: post-test probability 0.015
Pre-test 30%, positive strip: post-test probability 0.636
Pre-test 30%, negative strip: post-test probability 0.13


In [5]:
# 3. A simulation of 12 months of admissions screened for MRSA (seed 8): carriers, true positives, and false positives each month
import numpy as np

rng = np.random.default_rng(8)
adm = 2000
prev_mrsa = 0.03
se_m = 0.90
sp_m = 0.97
carriers = rng.binomial(adm, prev_mrsa, size=12)
true_pos = rng.binomial(carriers, se_m)
false_pos = rng.binomial(adm - carriers, 1 - sp_m)
positives = true_pos + false_pos
print("Monthly means - carriers:", round(carriers.mean(), 1), "; true positives:", round(true_pos.mean(), 1), "; false positives:", round(false_pos.mean(), 1), "; positives:", round(positives.mean(), 1))
print("Positives range from", positives.min(), "to", positives.max(), "; share of the positives that are true carriers:", round((true_pos / positives).mean(), 3))

# 4. Isolation bed-days: 7 days for a true carrier, 2 days for a false positive (until the culture result)
iso_days = true_pos * 7 + false_pos * 2
print("Isolation bed-days per month - mean:", round(iso_days.mean(), 1), "; maximum:", iso_days.max(), "=", round(iso_days.max() / 30, 1), "beds in a 30-day month")

Monthly means - carriers: 60.5 ; true positives: 55.2 ; false positives: 59.9 ; positives: 115.2
Positives range from 97 to 129 ; share of the positives that are true carriers: 0.48
Isolation bed-days per month - mean: 506.6 ; maximum: 568 = 18.9 beds in a 30-day month


### 9.7 Choosing a cutoff: the ROC curve and the area under it

*How does a continuous biomarker perform across cutoffs?*

In [6]:
# 1. Sensitivity and specificity of the biomarker at four cutoffs
from sklearn.metrics import roc_curve, roc_auc_score
from scipy import stats

x = d["marker_ng_ml"].to_numpy()
y = inf.to_numpy()
for c in [15, 30, 45, 60]:
    print("Cutoff", c, ": sensitivity", round((x[y] >= c).mean(), 3), "; specificity", round((x[~y] < c).mean(), 3))

# 2. The ROC curve, the area under it, and the Mann-Whitney count divided by n1 times n0
fpr, tpr, thr = roc_curve(y, x)
auc = roc_auc_score(y, x)
u = stats.mannwhitneyu(x[y], x[~y], alternative="two-sided").statistic
n1 = y.sum()
n0 = (~y).sum()
print("Retained thresholds:", len(thr), "; area under the curve:", round(auc, 3), "; Mann-Whitney count over n1 times n0:", round(u / (n1 * n0), 3))

# 3. The Hanley-McNeil standard error and the 95% interval of the area
q1 = auc / (2 - auc)
q2 = 2 * auc ** 2 / (1 + auc)
se_auc = np.sqrt((auc * (1 - auc) + (n1 - 1) * (q1 - auc ** 2) + (n0 - 1) * (q2 - auc ** 2)) / (n1 * n0))
print("Standard error of the area:", round(se_auc, 4), "; 95% interval:", round(auc - 1.96 * se_auc, 3), "to", round(auc + 1.96 * se_auc, 3))

Cutoff 15 : sensitivity 1.0 ; specificity 0.413
Cutoff 30 : sensitivity 0.713 ; specificity 0.825
Cutoff 45 : sensitivity 0.564 ; specificity 0.95
Cutoff 60 : sensitivity 0.372 ; specificity 0.984
Retained thresholds: 315 ; area under the curve: 0.876 ; Mann-Whitney count over n1 times n0: 0.876
Standard error of the area: 0.0235 ; 95% interval: 0.83 to 0.922


## Part B. Exercise solutions

### 9.15 Exercise 1. A troponin assay on arrival

Target condition: acute myocardial infarction in patients presenting with chest pain,
by a stated clinical definition. Index test: the point-of-care
troponin read at the fixed cutoff on arrival. Reference standard: the
final diagnosis by an adjudication panel using the laboratory
high-sensitivity troponin series, electrocardiography, imaging, and
30-day follow-up for events, applied to every patient regardless of
the arrival result. Population: consecutive patients presenting to the
emergency department with chest pain in whom discharge is being
considered, since that is where the test will be used. Ruling out
myocardial infarction alone does not establish that discharge is safe:
other acute coronary syndromes and other serious causes of chest pain
require clinical assessment. One
imperfection: the reference relies on a serial troponin whose own
cutoff and timing misclassify some early presenters, and a panel
that sees the arrival result has an incorporation problem. A study
in patients admitted to the coronary care unit contains the severe
end of the spectrum only, with no low-risk presenters; its sensitivity
would be inflated by spectrum effect and its specificity would be
estimated in the wrong patients, so it cannot say how many discharged
patients would be missed.

### 9.15 Exercise 2. The dipstick table

Reference positive 240: dipstick positive 204 (true positives),
negative 36 (false negatives). Reference negative 400: dipstick
positive 88 (false positives), negative 312 (true negatives).
Sensitivity $204/240=0.850$ and false-negative proportion $36/240=0.150$,
denominators the 240 culture-positive women. Specificity $312/400=0.780$
and false-positive proportion $88/400=0.220$, denominators the 400
culture-negative women. Positive predictive value $204/(204+88)=204/292
=0.699$, denominator the 292 dipstick-positive women; negative
predictive value $312/(312+36)=312/348=0.897$, denominator the 348
dipstick-negative women.

### 9.15 Exercise 3. Wilson intervals for the dipstick

Sensitivity 0.85 (95% CI 0.80-0.89), specificity 0.78 (95% CI
0.74-0.82). The specificity interval is narrower because it rests on
400 women against 240, and because 0.78 is nearer 0.5 than 0.85 is,
which works the other way but less strongly. A study of 640 women
still leaves sensitivity known only to within about five points
because only the culture-positive women contribute to it; the total
size is not the relevant number.

In [7]:
# 1. Wilson 95% intervals for the dipstick: 204 of 240 and 312 of 400
for k, n in [(204, 240), (312, 400)]:
    lo, hi = proportion_confint(k, n, alpha=0.05, method="wilson")
    print(k, "of", n, ":", round(k / n, 3), "(95% CI", round(lo, 3), "to", round(hi, 3), ")")

204 of 240 : 0.85 (95% CI 0.799 to 0.89 )
312 of 400 : 0.78 (95% CI 0.737 to 0.818 )


### 9.15 Exercise 4. The dipstick in two settings

By hand, in the nursing home with $p=0.55$:
$\mathrm{PPV}=0.55\times0.85/(0.55\times0.85+0.45\times0.22)
=0.4675/(0.4675+0.099)=0.825$ and
$\mathrm{NPV}=0.45\times0.78/(0.45\times0.78+0.55\times0.15)
=0.351/(0.351+0.0825)=0.810$. In telephone triage with $p=0.10$:
$\mathrm{PPV}=0.085/(0.085+0.198)=0.300$ and
$\mathrm{NPV}=0.702/(0.702+0.015)=0.979$.

In the nursing home a positive dipstick means infection in about five
residents of six, and a negative one still leaves a one-in-five chance;
in triage a positive dipstick means infection in fewer than one caller
in three, while a negative one is reassuring. The same strip is a
rule-in aid in one setting and a rule-out aid in the other, provided
the sensitivity and specificity transport to both populations.

In [8]:
# 1. The dipstick (sensitivity 0.85, specificity 0.78) in two settings with different prevalence
for prev in [0.55, 0.10]:
    ppv_prev = prev * 0.85 / (prev * 0.85 + (1 - prev) * (1 - 0.78))
    npv_prev = (1 - prev) * 0.78 / ((1 - prev) * 0.78 + prev * (1 - 0.85))
    print("Prevalence", prev, ": positive predictive value", round(ppv_prev, 3), "; negative predictive value", round(npv_prev, 3))

Prevalence 0.55 : positive predictive value 0.825 ; negative predictive value 0.81
Prevalence 0.1 : positive predictive value 0.3 ; negative predictive value 0.979


### 9.15 Exercise 5. D-dimer bands

Shares among the 300 with embolism and the 1,200 without: below 500,
$12/300=0.040$ and $720/1{,}200=0.600$, ratio 0.07; 500 to 999,
$78/300=0.260$ and $360/1{,}200=0.300$, ratio 0.87; 1,000 and above,
$210/300=0.700$ and $120/1{,}200=0.100$, ratio 7.0. Collapsed at 500:
sensitivity $288/300=0.960$, specificity $720/1{,}200=0.600$,
$\mathrm{LR}^{+}=0.960/0.400=2.4$ and $\mathrm{LR}^{-}=0.040/0.600
=0.067$. The binary version keeps the strong rule-out of a low result
but merges a nearly uninformative middle band (0.87) with a strongly
positive top band (7.0) into one weak positive ratio of 2.4.

### 9.15 Exercise 6. Two patients at 15\%

Tree of 1,000 patients: 150 with embolism, 850 without. A result of
1,200 ng/mL falls in the top band, obtained by $150\times0.700=105$
patients with embolism and $850\times0.100=85$ without: post-test
probability $105/190=0.553$. A result of 300 falls in the lowest band,
obtained by $150\times0.040=6$ with embolism and $850\times0.600=510$
without: $6/516=0.0116$. By odds: pre-test odds $0.15/0.85=0.176$;
times 7.0 gives 1.235, probability $1.235/2.235=0.553$; times 0.067
gives 0.0118, probability 0.0116. The two routes agree, as
Proposition 9.2 requires.

### 9.15 Exercise 7. Screening admissions for a resistant organism

By hand: $1{,}500\times0.02=30$ carriers, of whom $30\times0.92=27.6$
test positive; $1{,}470\times0.05=73.5$ false positives; PPV
$27.6/101.1=0.273$.

About 28 true and 76 false positives a month, 290 isolation bed-days
on average and up to 359 in the worst month, with 52% of the
bed-days generated by false positives: at 2% prevalence, a specificity
of 0.95 makes the program spend more on isolating non-carriers than
carriers.

In [9]:
# 1. Screening 1,500 admissions a month for a resistant organism (seed 8): carriers, true positives, false positives
rng = np.random.default_rng(8)
adm = 1500
prev_c = 0.02
se_c = 0.92
sp_c = 0.95
carriers = rng.binomial(adm, prev_c, size=12)
true_pos = rng.binomial(carriers, se_c)
false_pos = rng.binomial(adm - carriers, 1 - sp_c)

# 2. Isolation bed-days: 5 days for a true carrier, 2 days for a false positive
iso_days = true_pos * 5 + false_pos * 2
print("Monthly means - true positives:", round(true_pos.mean(), 1), "; false positives:", round(false_pos.mean(), 1), "; isolation bed-days:", round(iso_days.mean(), 1), "; worst month:", iso_days.max())
print("Share of the bed-days generated by false positives:", round((false_pos * 2).sum() / iso_days.sum(), 3))

Monthly means - true positives: 27.7 ; false positives: 75.6 ; isolation bed-days: 289.5 ; worst month: 359
Share of the bed-days generated by false positives: 0.522


### 9.15 Exercise 8. The biomarker by diabetes

By hand, at 25 ng/mL: 75 of 94 infected patients are at or above it,
sensitivity 0.798, and 812 of 1,106 uninfected below it, specificity
0.734; at 40: 57 of 94, sensitivity 0.606, and 1,025 of 1,106,
specificity 0.927.

The area is 0.861 (95% CI 0.789-0.933) in patients with diabetes and
0.882 (95% CI 0.821-0.943) without; the Mann-Whitney count divided by
$n_1n_0$ reproduces each. The point estimate is slightly lower in
patients with diabetes, but each group holds only about 50 infections.
Overlap of the separate intervals is not a formal test of their
difference and does not establish equal discrimination. The code
saves both subgroup curves in `figures/exercise08_roc.pdf`.

In [10]:
# 1. Sensitivity and specificity of the biomarker at cutoffs of 25 and 40 ng/mL (counts and proportions)
for c in [25, 40]:
    print("Cutoff", c, ": sensitivity", (x[inf] >= c).sum(), round((x[inf] >= c).mean(), 3), "; specificity", (x[~inf] < c).sum(), round((x[~inf] < c).mean(), 3))

# 2. The area under the ROC curve in patients with and without diabetes, with the Mann-Whitney check and the 95% interval
for g in ["yes", "no"]:
    m = (d["diabetes"] == g).to_numpy()
    yy = inf[m]
    xx = x[m]
    auc_g = roc_auc_score(yy, xx)
    n1_g = yy.sum()
    n0_g = (~yy).sum()
    u_g = stats.mannwhitneyu(xx[yy], xx[~yy]).statistic
    q1_g = auc_g / (2 - auc_g)
    q2_g = 2 * auc_g ** 2 / (1 + auc_g)
    se_g = np.sqrt((auc_g * (1 - auc_g) + (n1_g - 1) * (q1_g - auc_g ** 2) + (n0_g - 1) * (q2_g - auc_g ** 2)) / (n1_g * n0_g))
    print("Diabetes", g + ": infected", n1_g, "; not infected", n0_g, "; area", round(auc_g, 3), "; Mann-Whitney", round(u_g / (n1_g * n0_g), 3), "; 95% interval", round(auc_g - 1.96 * se_g, 3), "to", round(auc_g + 1.96 * se_g, 3))

Cutoff 25 : sensitivity 75 0.798 ; specificity 812 0.734
Cutoff 40 : sensitivity 57 0.606 ; specificity 1025 0.927
Diabetes yes: infected 43 ; not infected 267 ; area 0.861 ; Mann-Whitney 0.861 ; 95% interval 0.789 to 0.933
Diabetes no: infected 51 ; not infected 839 ; area 0.882 ; Mann-Whitney 0.882 ; 95% interval 0.821 to 0.943


### 9.15 Exercise 9. A cost ratio of five

With a missed infection five times as costly as a false alarm at 7.8%
prevalence, $w=2.35$ and the cost-minimizing cutoff is 44.9 ng/mL,
sensitivity 0.574 and specificity 0.950, higher than Youden's 32.3
because the odds against infection (11.8) still exceed the cost ratio
(5), so false alarms dominate the total cost. Among referred wounds at
30%, $w=0.70/(0.30\times5)=0.467$, and the cutoff falls to 15.4, where
sensitivity is 1.0 and specificity 0.432: with infection common and
costly, nearly every wound should be reviewed, and the biomarker adds
little to a policy of reviewing all.

In [11]:
# 1. The cutoff that maximizes the weighted sensitivity when a false positive costs one fifth of a missed case
fpr, tpr, thr = roc_curve(inf, x)
prev = inf.mean()
w = (1 - prev) / (prev * 5)
k = np.argmax(tpr - w * fpr)
j = np.argmax(tpr - fpr)
print("Weight:", round(w, 2), "; cutoff:", round(thr[k], 1), "; sensitivity:", round(tpr[k], 3), "; specificity:", round(1 - fpr[k], 3))

# 2. The Youden cutoff (equal costs)
print("Youden cutoff:", round(thr[j], 1), "; sensitivity:", round(tpr[j], 3), "; specificity:", round(1 - fpr[j], 3))

# 3. The same with a prevalence of 30%
w2 = 0.70 / (0.30 * 5)
k2 = np.argmax(tpr - w2 * fpr)
print("Weight:", round(w2, 3), "; cutoff:", round(thr[k2], 1), "; sensitivity:", round(tpr[k2], 3), "; specificity:", round(1 - fpr[k2], 3))

Weight: 2.35 ; cutoff: 44.9 ; sensitivity: 0.574 ; specificity: 0.95
Youden cutoff: 32.3 ; sensitivity: 0.713 ; specificity: 0.855
Weight: 0.467 ; cutoff: 15.4 ; sensitivity: 1.0 ; specificity: 0.432


### 9.15 Exercise 10. Three cancer-test evaluations

(a) Two-gate case-control design with an extreme spectrum: metastatic
cases and healthy donors. Spectrum effect inflates both sensitivity
(easy cases) and specificity (easy non-cases); QUADAS-2 domain 1,
patient selection (“was a case-control design avoided?” and
applicability). (b) Single-gate consecutive design with differential
verification: positives biopsied, negatives followed by telephone. A
telephone follow-up misses cancers among test negatives, so false
negatives are undercounted and sensitivity is inflated; specificity is
inflated as well under this mechanism: missed cancers among the test
negatives are relabeled as true negatives, increasing the numerator
and denominator of specificity while false positives are unchanged.
QUADAS-2 domain 3 concerns the telephone reference's accuracy;
domain 4, flow and timing, asks whether all patients received the same
reference standard. Other reference errors could change the direction. (c) Single-gate consecutive
design with full biopsy verification but no blinding of the
pathologist: knowledge of the blood test result can pull the reference
toward agreement, inflating both sensitivity and specificity by
review bias. QUADAS-2 domain 3, reference standard (“were the
reference standard results interpreted without knowledge of the index
test results?”).

### 9.15 Exercise 11. Sample size for a rapid influenza test

Sensitivity needs 137 patients with influenza, which at 12%
prevalence means enrolling 1,142 febrile patients; specificity needs
203 without influenza, which 231 enrollments supply. The study must
enroll the larger, 1,142. In a mild season at 6%, the influenza
requirement is unchanged at 137 but the enrollment doubles to 2,284,
because half as many of those enrolled contribute to the sensitivity.
These totals yield the required group sizes in expectation, not with
certainty; actual accrual of both groups must be monitored.

In [12]:
# 1. Sample size for a rapid influenza test: patients with influenza for the sensitivity, and the enrollment at 12% prevalence
z = 1.96
n_inf = z ** 2 * 0.85 * 0.15 / 0.06 ** 2
print("Patients with influenza:", round(n_inf, 1), "->", int(np.ceil(n_inf)), "; febrile patients to enroll at 12% prevalence:", int(np.ceil(np.ceil(n_inf) / 0.12)))

# 2. Patients without influenza for the specificity, and the enrollment
n_not = z ** 2 * 0.95 * 0.05 / 0.03 ** 2
print("Patients without influenza:", round(n_not, 1), "->", int(np.ceil(n_not)), "; febrile patients to enroll:", int(np.ceil(np.ceil(n_not) / 0.88)))

# 3. In a mild season at 6% prevalence
print("Febrile patients to enroll at 6% prevalence:", int(np.ceil(np.ceil(n_inf) / 0.06)))

Patients with influenza: 136.1 -> 137 ; febrile patients to enroll at 12% prevalence: 1142
Patients without influenza: 202.8 -> 203 ; febrile patients to enroll: 231
Febrile patients to enroll at 6% prevalence: 2284


### 9.15 Exercise 12. Four short questions

(a) Serial strategy under independence: sensitivity
$0.90\times0.86=0.774$; specificity $1-(1-0.50)(1-0.95)=1-0.025
=0.975$. The observed sensitivity may differ because the children the
score misses (few symptoms) are also those with a low bacterial load
whom the antigen test misses, so the second test excludes fewer of the
score-positive infected children than independence predicts, and the
observed sensitivity is likely higher than 0.774.

(b) Annual screening in the synthetic cohort.

Annual screening detects 820 tumors (69%) instead of 644, with a mean
lead time of 2.96 years. Earlier detection of the same tumor would
lengthen its lead time. The slightly shorter average here concerns
a different detected group, including more fast tumors, and different
random screening results; it is not evidence of later detection under
more frequent screening.
Five-year survival is 65.9% among screen-detected and 37.8% among
clinically detected cases; the 688 deaths within 20 years are
unchanged, because the program moves diagnoses and nothing else.

(c) Number needed to screen $10{,}000/25=400$ people for 10 years per
death prevented; $60/25=2.4$ overdiagnosed cases per death prevented.
The numbers alone do not establish whether the test is acceptable to
the population or whether adequate diagnostic and treatment facilities
exist. Both are Wilson-Jungner criteria.

(d) Treatment threshold $0.04/(0.04+0.08)=0.333$. Test threshold:
the pre-test probability whose odds times 6 reach odds of
$0.333/0.667=0.5$, that is, odds $0.0833$ and probability $0.077$.
Test-treatment threshold: the pre-test probability whose odds times
0.2 still equal 0.5, odds 2.5 and probability $0.714$. The test is worth running
for pre-test probabilities between about 8% and 71% under the
stated assumption of no test cost, harm, or delay; below 8% even
a positive result leaves the probability under the treatment
threshold, and above 71% even a negative result leaves it above.

In [13]:
# 1. Annual screening in the synthetic cohort of 10,000 women followed for 20 years (seed 8): each year, 85% of the tumors in their detectable phase are found
import pandas as pd

s = pd.read_csv("data/screening_cohort.csv")
c = s[s["cancer"] == "yes"].copy()
rng = np.random.default_rng(8)
onset = c["t_onset"].to_numpy()
t_clin = c["t_clinical"].to_numpy()
t_death = c["t_death"].to_numpy()
t_screen = np.full(len(c), np.nan)
for sc in np.arange(1.0, 20.1, 1.0):
    in_phase = (onset <= sc) & (sc < t_clin) & np.isnan(t_screen)
    hit = in_phase & (rng.random(len(c)) < 0.85)
    t_screen[hit] = sc

# 2. Detected tumors, their share, and the mean lead time (years)
det = ~np.isnan(t_screen)
t_dx = np.where(det, t_screen, t_clin)
print("Screen-detected tumors:", det.sum(), "; share:", round(det.mean(), 3), "; mean lead time:", round((t_clin - t_screen)[det].mean(), 2), "years")

# 3. Survival beyond 5 years from diagnosis among the screen-detected and the clinically detected, and deaths within 20 years
print("Survival beyond 5 years - screen-detected:", round(((t_death - t_dx) > 5)[det].mean(), 3), "; clinically detected:", round(((t_death - t_dx) > 5)[~det].mean(), 3), "; deaths within 20 years:", int((t_death <= 20).sum()))

# 4. The treatment threshold from the harm of 0.04 of treating a patient without the condition and the benefit of 0.08, and the test threshold
treat_thr = 0.04 / (0.04 + 0.08)
print("Treatment threshold:", round(treat_thr, 3), "; test threshold with a likelihood ratio of 6:", round(treat_thr / (treat_thr + (1 - treat_thr) * 6), 3), "; with 0.2:", round(treat_thr / (treat_thr + (1 - treat_thr) * 0.2), 3))

Screen-detected tumors: 820 ; share: 0.69 ; mean lead time: 2.96 years
Survival beyond 5 years - screen-detected: 0.659 ; clinically detected: 0.378 ; deaths within 20 years: 688
Treatment threshold: 0.333 ; test threshold with a likelihood ratio of 6: 0.077 ; with 0.2: 0.714


---
*Synthetic data (`generate_data.py`, seed 8). Teaching values only; nothing here is an estimate for a real service.*